# Notebook 10 — FEP Validation: Barnase-Barstar Ala59→Gly

This notebook validates the alchemical free energy perturbation (FEP)
implementation against the experimentally characterized barnase-barstar
system using the **Schreiber & Fersht (1995)** mutational database.

## Thermodynamic Cycle

```
WT complex ───(DG_bind,WT)───▶ WT unbound
    │                                │
    │ DG_alch,complex                │ DG_alch,free
    │                                │
    ▼                                ▼
Mut complex ──(DG_bind,Mut)──▶ Mut unbound
```

$$\Delta\Delta G = \Delta G_{\text{alch,complex}} - \Delta G_{\text{alch,free}}$$

## Benchmark Mutation

- **Mutation:** Ala59 → Gly in barnase
- **Experimental DDG:** +2.0 kcal/mol (+8.4 kJ/mol) — destabilizes binding
- **Reference:** Schreiber & Fersht, *J. Mol. Biol.*, 248, 478–486 (1995)
- **Acceptance criterion:** |DDG_computed - DDG_experimental| < 1 kcal/mol

In [ ]:
# Cell 1 — Environment Setup
!pip install openmm openmmtools pymbar mdtraj numpy matplotlib

import openmm
import openmmtools
import pymbar

print(f"OpenMM version: {openmm.__version__}")
print(f"openmmtools version: {openmmtools.__version__}")
print(f"pymbar version: {pymbar.__version__}")

platforms = [openmm.Platform.getPlatform(i).getName() for i in range(openmm.Platform.getNumPlatforms())]
print(f"Available platforms: {platforms}")

In [ ]:
# Cell 2 — Project Setup & Imports
import sys
from pathlib import Path

# For Colab: mount Drive and set up project root
USE_COLAB = False
if USE_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/SPINK7_KLK5')
else:
    PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import FEPConfig, BOLTZMANN_KJ, KCAL_TO_KJ
from src.simulate.fep import generate_lambda_schedule, create_alchemical_system, run_fep_campaign
from src.analyze.fep import compute_delta_g_mbar, compute_delta_delta_g
from src.simulate.platform import select_platform

FEP_OUTPUT = PROJECT_ROOT / 'data' / 'analysis' / 'fep'
FEP_OUTPUT.mkdir(parents=True, exist_ok=True)
print(f"FEP output directory: {FEP_OUTPUT}")

In [ ]:
# Cell 3 — Load Barnase-Barstar Complex (PDB: 1BRS)
import mdtraj as md
from openmm.app import PDBFile, ForceField, Modeller
from openmm import unit

# Load the barnase-barstar complex
pdb_path = PROJECT_ROOT / 'data' / 'pdb' / 'raw' / '1BRS.pdb'
pdb = PDBFile(str(pdb_path))
print(f"Loaded {pdb_path.name}: {pdb.topology.getNumAtoms()} atoms")

# Set up force field
forcefield = ForceField('amber14-all.xml', 'amber14/tip3p.xml')

# Prepare with Modeller: add hydrogens, solvate
modeller = Modeller(pdb.topology, pdb.positions)
modeller.addHydrogens(forcefield, pH=7.4)
modeller.addSolvent(
    forcefield,
    model='tip3p',
    padding=1.0 * unit.nanometers,
    ionicStrength=0.15 * unit.molar,
)
print(f"Solvated system: {modeller.topology.getNumAtoms()} atoms")

In [ ]:
# Cell 4 — Identify Ala59 mutant atoms
# Find atom indices for Ala59 in barnase (chain A)

mutant_residue_number = 59
mutant_chain_id = 'A'  # barnase chain

mutant_atom_indices = []
for atom in modeller.topology.atoms():
    if (atom.residue.chain.id == mutant_chain_id
            and atom.residue.id == str(mutant_residue_number)
            and atom.residue.name == 'ALA'):
        mutant_atom_indices.append(atom.index)

print(f"Ala{mutant_residue_number} atom indices ({len(mutant_atom_indices)} atoms): {mutant_atom_indices}")
assert len(mutant_atom_indices) > 0, f"No atoms found for Ala{mutant_residue_number}"

In [ ]:
# Cell 5 — Create system and FEP config
system_complex = forcefield.createSystem(
    modeller.topology,
    nonbondedMethod=openmm.app.PME,
    nonbondedCutoff=1.0 * unit.nanometers,
    constraints=openmm.app.HBonds,
)

config = FEPConfig(
    n_lambda_windows=20,
    per_window_duration_ns=2.0,
    temperature_k=310.0,
)

print(f"System created: {system_complex.getNumParticles()} particles")
print(f"Lambda windows: {config.n_lambda_windows}")
print(f"Per-window duration: {config.per_window_duration_ns} ns")

In [ ]:
# Cell 6 — Run Complex Leg FEP Campaign
# GPU offload: this requires CUDA platform (~2-3 GPU hours on A100)

complex_output = FEP_OUTPUT / 'barnase_barstar_complex'
complex_results = run_fep_campaign(
    system=system_complex,
    positions=modeller.positions,
    mutant_atom_indices=mutant_atom_indices,
    config=config,
    output_dir=complex_output,
)

print(f"Complex leg complete: energy matrix shape {complex_results['energy_matrix'].shape}")

In [ ]:
# Cell 7 — Prepare Free-Solution Barnase (No Barstar)
# Extract barnase chain only for the free-solution leg

from openmm.app import PDBFile as PDBFileWriter

# Identify barnase chain atoms
barnase_atom_indices_free = []
for atom in pdb.topology.atoms():
    if atom.residue.chain.id == mutant_chain_id:
        barnase_atom_indices_free.append(atom.index)

# Create a sub-topology with only barnase
modeller_free = Modeller(pdb.topology, pdb.positions)
# Remove non-barnase chains
chains_to_remove = [
    c for c in modeller_free.topology.chains()
    if c.id != mutant_chain_id
]
modeller_free.delete(chains_to_remove)
modeller_free.addHydrogens(forcefield, pH=7.4)
modeller_free.addSolvent(
    forcefield,
    model='tip3p',
    padding=1.0 * unit.nanometers,
    ionicStrength=0.15 * unit.molar,
)

system_free = forcefield.createSystem(
    modeller_free.topology,
    nonbondedMethod=openmm.app.PME,
    nonbondedCutoff=1.0 * unit.nanometers,
    constraints=openmm.app.HBonds,
)

# Re-identify Ala59 in free-solution barnase topology
free_mutant_indices = []
for atom in modeller_free.topology.atoms():
    if (atom.residue.id == str(mutant_residue_number)
            and atom.residue.name == 'ALA'):
        free_mutant_indices.append(atom.index)

print(f"Free barnase system: {system_free.getNumParticles()} particles")
print(f"Ala{mutant_residue_number} free-leg atom indices: {len(free_mutant_indices)} atoms")

In [ ]:
# Cell 8 — Run Free-Solution Leg FEP Campaign
# GPU offload: ~2-3 GPU hours on A100

free_output = FEP_OUTPUT / 'barnase_free'
free_results = run_fep_campaign(
    system=system_free,
    positions=modeller_free.positions,
    mutant_atom_indices=free_mutant_indices,
    config=config,
    output_dir=free_output,
)

print(f"Free leg complete: energy matrix shape {free_results['energy_matrix'].shape}")

In [ ]:
# Cell 9 — MBAR Analysis: Compute DG for Both Legs
import numpy as np

dg_complex = compute_delta_g_mbar(
    complex_results['energy_matrix'],
    complex_results['n_samples_per_state'],
    temperature_k=310.0,
)

dg_free = compute_delta_g_mbar(
    free_results['energy_matrix'],
    free_results['n_samples_per_state'],
    temperature_k=310.0,
)

print(f"DG_alch,complex = {dg_complex['delta_g_kj_mol']:.2f} \u00b1 {dg_complex['delta_g_std_kj_mol']:.2f} kJ/mol")
print(f"DG_alch,free    = {dg_free['delta_g_kj_mol']:.2f} \u00b1 {dg_free['delta_g_std_kj_mol']:.2f} kJ/mol")

In [ ]:
# Cell 10 — Compute DDG and Validate Against Experiment
ddg = compute_delta_delta_g(dg_complex, dg_free)

DDG_EXPERIMENTAL_KCAL = 2.0  # Schreiber & Fersht, 1995
DDG_EXPERIMENTAL_KJ = DDG_EXPERIMENTAL_KCAL * KCAL_TO_KJ
ACCEPTANCE_KCAL = 1.0  # kcal/mol acceptance threshold

discrepancy_kcal = abs(ddg['delta_delta_g_kcal_mol'] - DDG_EXPERIMENTAL_KCAL)

print('=' * 60)
print('FEP Validation: Barnase Ala59 -> Gly')
print('=' * 60)
print(f"DDG computed:     {ddg['delta_delta_g_kcal_mol']:.2f} \u00b1 {ddg['delta_delta_g_std_kcal_mol']:.2f} kcal/mol")
print(f"DDG experimental: {DDG_EXPERIMENTAL_KCAL:.2f} kcal/mol")
print(f"Discrepancy:      {discrepancy_kcal:.2f} kcal/mol")
print(f"Threshold:        {ACCEPTANCE_KCAL:.2f} kcal/mol")
print(f"Result:           {'PASSED' if discrepancy_kcal < ACCEPTANCE_KCAL else 'FAILED'}")
print('=' * 60)

# Save results
np.savez(
    FEP_OUTPUT / 'barnase_A59G_ddg.npz',
    delta_delta_g_kj_mol=ddg['delta_delta_g_kj_mol'],
    delta_delta_g_kcal_mol=ddg['delta_delta_g_kcal_mol'],
    delta_delta_g_std_kj_mol=ddg['delta_delta_g_std_kj_mol'],
    delta_delta_g_std_kcal_mol=ddg['delta_delta_g_std_kcal_mol'],
    ddg_experimental_kcal_mol=DDG_EXPERIMENTAL_KCAL,
)

## Next Steps

If validation passes (|DDG_computed - DDG_experimental| < 1 kcal/mol):

1. Proceed to **Notebook 11** for SPINK7-KLK5 mutagenesis scanning.
2. Target key interface residues: RSL P1 Arg, P2 hydrophobic, disulfide Cys.
3. Characterize EoE-associated *SPINK7* variants.

### Hardware Requirements

- **This notebook:** ~4–6 GPU hours on NVIDIA A100
- **Each mutation scan:** ~4–6 GPU hours per thermodynamic cycle

### References

[1] Schreiber & Fersht, *J. Mol. Biol.*, 248, 478–486 (1995)  
[2] Bennett, *J. Comput. Phys.*, 22, 245–268 (1976)  
[3] Shirts & Chodera, *J. Chem. Phys.*, 129, 124105 (2008)